# Step 2: Classifier tool

Trains a Random Forest on structured MSHA fields to predict `DEGREE_INJURY_CD` (10 classes). Uses `select_classifier_features()` to avoid leakage columns.

**CLI equivalent:** `python -m src.tools.run_classifier`

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.tools.classifier import CLASSIFIER_REPORT_JSON, train_and_evaluate

In [ ]:
report = train_and_evaluate(save_model=True)
primary = report["evaluations"][0]
print(f"Accuracy: {primary['accuracy']:.4f}")
print(f"Macro F1: {primary['macro_f1']:.4f}")

In [ ]:
import pandas as pd

with CLASSIFIER_REPORT_JSON.open() as f:
    report = json.load(f)

primary = report["evaluations"][0]
recall_rows = [
    {"code": k, "recall": v, "support": primary["per_class_support"][k]}
    for k, v in primary["per_class_recall"].items()
]
pd.DataFrame(recall_rows).sort_values("code")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = primary["confusion_matrix"]
labels = primary["labels"]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=False, fmt="d", xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion matrix (stratified holdout)")
plt.tight_layout()
plt.show()